# BNO055 binary file explorer

Poke at the per-stream `.bin` files written for the BNO055 IMU under one NeuropixelsV2 recording.
Each stream is its own file (no embedded timestamps); timing lives in the `Clock` / `HubSyncCounter` files.
Here we look at the `_84` (sleep) block.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

data_dir = Path("/Users/willreith/Documents/phd/aeon/data/ephys/abcEphysPilot04/26-05-19T14-13-54/NeuropixelsV2")
block = "84"

for p in sorted(data_dir.glob(f"*Bno055*_{block}.bin")):
    print(f"{p.name:45s} {p.stat().st_size:>10d} bytes")

In [ ]:
# stream -> (dtype, n components per sample), inferred from file sizes + content
streams = {
    "Clock":              ("<u8", 1),
    "HubSyncCounter":     ("<u8", 1),
    "Euler":              ("<f4", 3),
    "Quaternion":         ("<f4", 4),
    "GravityVector":      ("<f4", 3),
    "LinearAcceleration": ("<f4", 3),
}

def load(name):
    dtype, ncomp = streams[name]
    arr = np.fromfile(data_dir / f"NeuropixelsV2_Bno055_{name}_{block}.bin", dtype=dtype)
    return arr.reshape(-1, ncomp)

for name in streams:
    a = load(name)
    print(f"{name:20s} shape={str(a.shape):12s} dtype={a.dtype}  head={a[0]}")

In [ ]:
# Clock is a free-running ONIX hardware counter @ 250 MHz
HW_CLOCK_HZ = 250e6

clk = load("Clock").ravel()
dt = np.diff(clk)
t = (clk - clk[0]) / HW_CLOCK_HZ

print("n samples         :", clk.size)
print("duration (s)      :", t[-1])
print("median dt (counts):", np.median(dt))
print("effective rate Hz :", HW_CLOCK_HZ / np.median(dt))
print("HubSyncCounter head:", load("HubSyncCounter").ravel()[:5])

# physical sanity checks
quat = load("Quaternion")
grav = load("GravityVector")
print("quat norm  mean/std:", np.linalg.norm(quat, axis=1).mean(), np.linalg.norm(quat, axis=1).std())
print("|gravity|  mean    :", np.linalg.norm(grav, axis=1).mean())
print("Euler quant step   :", np.min(np.diff(np.unique(load("Euler")[:, 0]))))

In [ ]:
# fusion outputs (Euler/Quaternion/Gravity) update slower than they are sampled;
# linear acceleration changes essentially every sample
def frac_repeated(a):
    return np.all(a[1:] == a[:-1], axis=1).mean()

for name in ["Euler", "Quaternion", "GravityVector", "LinearAcceleration"]:
    print(f"{name:20s} frac identical-to-prev: {frac_repeated(load(name)):.3f}")

In [ ]:
euler = load("Euler")
lin = load("LinearAcceleration")

fig, ax = plt.subplots(3, 1, sharex=True, figsize=(11, 7))
for i, lbl in enumerate(["heading", "roll", "pitch"]):
    ax[0].plot(t, euler[:, i], lw=0.6, label=lbl)
ax[0].set_ylabel("Euler (deg)"); ax[0].legend(loc="upper right", ncol=3)
for i, lbl in enumerate(["x", "y", "z"]):
    ax[1].plot(t, lin[:, i], lw=0.6, label=lbl)
ax[1].set_ylabel("lin accel (m/s\u00b2)"); ax[1].legend(loc="upper right", ncol=3)
for i, lbl in enumerate(["x", "y", "z"]):
    ax[2].plot(t, grav[:, i], lw=0.6, label=lbl)
ax[2].set_ylabel("gravity (m/s\u00b2)"); ax[2].set_xlabel("time (s)"); ax[2].legend(loc="upper right", ncol=3)
fig.tight_layout()